# 02 – Mô hình truyền thống: LDA (A1), LSA + K-Means (A2)
**Đầu ra:** `results.csv`, `topic_words.csv`, cột `lda` và `lsa_kmeans` của `doc_topics.csv`, `models/lda/`

In [1]:
import sys; sys.path.append("..")
from src.config import *

In [2]:
import time
import pandas as pd
from src.preprocess import load_articles, get_tokens
from src.metrics import evaluate, make_row, save_results
from src.baselines import run_lda, run_lsa_kmeans, save_lda

df = load_articles()
texts = get_tokens(df)              # list[list[str]]: cho LDA và C_v
text_tok = df["text_tok"].tolist()  # list[str]: cho LSA
y = df["true_label"].values         # nhãn thật, chỉ dùng để chấm điểm
print("Số bài:", len(df))

Số bài: 1800


In [3]:
rows = []
for seed in SEEDS:
    t0 = time.perf_counter()
    labels, topic_words, _, _ = run_lda(texts, seed=seed)
    time_s = time.perf_counter() - t0
    scores = evaluate(labels, y, topic_words, texts)
    rows.append(make_row("lda", "bow", "k6", "", seed, scores, time_s))
    print(f"LDA seed={seed}: NMI={scores['nmi']:.3f}  C_v={scores['cv']:.3f}  ({time_s:.1f}s)")

for n_dims in SVD_DIMS:
    for seed in SEEDS:
        t0 = time.perf_counter()
        labels, topic_words, _ = run_lsa_kmeans(text_tok, n_dims, seed=seed)
        time_s = time.perf_counter() - t0
        scores = evaluate(labels, y, topic_words, texts)
        rows.append(make_row("lsa_kmeans", "tfidf_svd", "k6", f"svd_dim={n_dims}", seed, scores, time_s))
        print(f"LSA {n_dims} chiều, seed={seed}: NMI={scores['nmi']:.3f}  C_v={scores['cv']:.3f}")

save_results(rows)

LDA seed=42: NMI=0.509  C_v=0.707  (24.2s)
LDA seed=43: NMI=0.478  C_v=0.635  (21.4s)
LDA seed=44: NMI=0.522  C_v=0.695  (19.9s)
LSA 10 chiều, seed=42: NMI=0.615  C_v=0.739
LSA 10 chiều, seed=43: NMI=0.669  C_v=0.793
LSA 10 chiều, seed=44: NMI=0.629  C_v=0.734
LSA 20 chiều, seed=42: NMI=0.676  C_v=0.781
LSA 20 chiều, seed=43: NMI=0.680  C_v=0.775
LSA 20 chiều, seed=44: NMI=0.672  C_v=0.782
LSA 50 chiều, seed=42: NMI=0.642  C_v=0.793
LSA 50 chiều, seed=43: NMI=0.643  C_v=0.795
LSA 50 chiều, seed=44: NMI=0.668  C_v=0.788


In [4]:
res = pd.read_csv(RESULTS_CSV, keep_default_na=False)
res = res[res["method"].isin(["lda", "lsa_kmeans"])]
cols = ["ari", "nmi", "purity", "cv", "diversity", "time_s"]
summary = res.groupby(["method", "param"])[cols].agg(["mean", "std"]).round(3)
summary

ari           nmi        purity            cv         \
                        mean    std   mean    std   mean    std   mean    std   
method     param                                                                
lda                    0.448  0.032  0.503  0.022  0.652  0.021  0.679  0.039   
lsa_kmeans svd_dim=10  0.592  0.027  0.638  0.028  0.769  0.041  0.755  0.033   
           svd_dim=20  0.627  0.004  0.676  0.004  0.822  0.002  0.780  0.004   
           svd_dim=50  0.572  0.046  0.651  0.015  0.786  0.027  0.792  0.004   

                      diversity         time_s         
                           mean    std    mean    std  
method     param                                       
lda                       0.922  0.025  21.812  2.170  
lsa_kmeans svd_dim=10     0.967  0.000   2.606  1.844  
           svd_dim=20     0.967  0.000   1.658  0.027  
           svd_dim=50     0.978  0.010   2.091  0.102

In [5]:
cv_by_dim = res[res["method"] == "lsa_kmeans"].groupby("param")["cv"].mean()
print(cv_by_dim.round(3))
best_dim = int(cv_by_dim.idxmax().split("=")[1])
print("Số chiều SVD được chọn (C_v cao nhất):", best_dim)

param
svd_dim=10    0.755
svd_dim=20    0.780
svd_dim=50    0.792
Name: cv, dtype: float64
Số chiều SVD được chọn (C_v cao nhất): 50


In [6]:
lda_labels, lda_words, lda, dictionary = run_lda(texts, seed=SEED)
lsa_labels, lsa_words, _ = run_lsa_kmeans(text_tok, best_dim, seed=SEED)
save_lda(lda, dictionary)

def topic_rows(method, labels, words):
    counts = pd.Series(labels).value_counts()
    return [{"method": method, "topic": t, "n_docs": int(counts.get(t, 0)), "words": " ".join(w)}
            for t, w in enumerate(words)]

tw = pd.DataFrame(topic_rows("lda", lda_labels, lda_words) + topic_rows("lsa_kmeans", lsa_labels, lsa_words))
if TOPIC_WORDS_CSV.exists():
    old = pd.read_csv(TOPIC_WORDS_CSV)
    tw = pd.concat([old[~old["method"].isin(["lda", "lsa_kmeans"])], tw], ignore_index=True)
tw.to_csv(TOPIC_WORDS_CSV, index=False, encoding="utf-8")
pd.set_option("display.max_colwidth", None)
tw

,method,topic,n_docs,words
0,lda,0,278,việt_nam đội trận bóng đội_tuyển giải cầu_thủ hlv thi_đấu thắng
1,lda,1,159,nvidia công_ty huang ceo mỹ lãnh_đạo vụ jensen chip công_nghệ
2,lda,2,281,đồng giá tỷ triệu giảm usd tăng tiền thị_trường đầu_tư
3,lda,3,398,bác_sĩ bệnh dùng điều_trị người_bệnh bệnh_viện giảm tăng tình_trạng y_tế
4,lda,4,349,việt_nam doanh_nghiệp công_nghệ nước kinh_tế đầu_tư xây_dựng sản_xuất sản_phẩm nghiên_cứu
5,lda,5,335,phim diễn_viên chương_trình văn_hóa khán_giả tuổi gia_đình ăn bộ việt_nam
6,lsa_kmeans,0,272,trận đội đội_tuyển bóng cầu_thủ hlv việt_nam u17 giải thắng
7,lsa_kmeans,1,198,nvidia usd đồng huang giá cổ_phiếu tỷ công_ty ceo thị_trường
8,lsa_kmeans,2,288,bác_sĩ điều_trị người_bệnh bệnh bệnh_viện ung_thư ăn thuốc đau thận
9,lsa_kmeans,3,589,doanh_nghiệp việt_nam công_nghệ dự_án đầu_tư đất dân kinh_tế tỉnh nước


In [7]:
dt = pd.read_csv(DOC_TOPICS_CSV) if DOC_TOPICS_CSV.exists() else df[["id", "true_label"]].copy()
dt["lda"] = lda_labels
dt["lsa_kmeans"] = lsa_labels
dt.to_csv(DOC_TOPICS_CSV, index=False, encoding="utf-8")

In [8]:
for name, lab in [("LDA", lda_labels), (f"LSA ({best_dim} chiều)", lsa_labels)]:
    print(name)
    display(pd.crosstab(pd.Series(lab, name="Chủ đề"), df["true_label"].map(TOPICS).rename("Nhãn thật")))

LDA


Nhãn thật,Công nghệ,Giải trí,Kinh tế,Sức khỏe,Thể thao,Thời sự
Chủ đề,,,,,,
0,2,0,1,0,267,8
1,54,7,32,1,0,65
2,10,4,143,5,8,111
3,98,0,10,268,6,16
4,128,7,108,9,8,89
5,8,282,6,17,11,11


LSA (50 chiều)


Nhãn thật,Công nghệ,Giải trí,Kinh tế,Sức khỏe,Thể thao,Thời sự
Chủ đề,,,,,,
0,2,1,0,1,268,0
1,48,6,137,0,6,1
2,0,1,0,270,6,11
3,87,29,150,23,15,285
4,1,263,6,5,5,2
5,162,0,7,1,0,1


## Phân tích lỗi LDA và BERTopic
(Làm sau khi có cột `bertopic_k6` trong `doc_topics.csv`)